# 02. 上下文隔离与 Prompt Caching

两个互补的省 token 手段：
- **上下文隔离**：sub-agent 只拿到它需要的切片，主 agent 只回收结论 — 中间过程永不进入主 context
- **Prompt caching**：前缀不变的部分被服务商缓存（DeepSeek 自动命中），重复前缀只付一小部分费用

> **检查点**：能解释为什么「把时间戳放在 system prompt 开头」会摧毁 cache 命中率。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()

🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# Sub-agent 上下文隔离：orchestrator 只传切片，只收结论
BIG_CONTEXT = """[项目背景 5000 字省略] 数据库是 PostgreSQL 15，ORM 用 SQLAlchemy 2.0。
[前端规范 3000 字省略] [部署文档 4000 字省略] 缓存层是 Redis 7，session 存 hash。"""

def orchestrator(task: str) -> str:
    # 反模式：把整个 BIG_CONTEXT 塞给 sub-agent
    # 正确：只切出相关片段
    if "redis" in task.lower() or "session" in task.lower():
        slice_ = "缓存层是 Redis 7，session 存 hash。"
    else:
        slice_ = "数据库是 PostgreSQL 15，ORM 用 SQLAlchemy 2.0。"
    print(f"主 context: {len(BIG_CONTEXT)} chars → sub-agent 只收到切片: {len(slice_)} chars")
    conclusion = chat(client, f"基于这个事实：{slice_}\n回答：{task}", max_tokens=150)
    return conclusion  # 只有结论回到主 context，sub-agent 的中间推理不回流

print(orchestrator("session 数据应该用 Redis 的什么数据结构存？"))

主 context: 117 chars → sub-agent 只收到切片: 28 chars


根据你提供的事实，session 数据应该使用 Redis 的 **Hash（哈希）** 数据结构来存储。因为哈希结构适合存储对象类型的字段，可以方便地存取 session 中的多个属性（如用户ID、过期时间等），且 Redis 7 对 Hash 操作做了进一步优化（如


## Prompt Caching 命中条件

| 做法 | 命中？ | 原因 |
|------|-------|------|
| 固定 system prompt 前缀 | ✅ | 前缀完全一致 |
| 开头插入当前时间戳 | ❌ | 每轮前缀都变 |
| 对话历史 append-only | ✅ | 旧前缀保持不变 |
| 在中间编辑历史消息 | ❌ | 编辑点之后全部失效 |
| 动态内容（用户画像等）放尾部 | ✅ | 只影响最后一段 |

DeepSeek 的 context caching 自动生效，无需手动开启 — 但**只有前缀稳定才命中**。

In [3]:
# 观察 cache 命中：同一长前缀连续调用两次，对比 usage 中的 cache 字段
import time

LONG_PREFIX = "你是资深 DBA。" + "数据库规范：索引必须有注释；查询必须带 LIMIT；" * 50

def timed_call(tag: str):
    if client is None:
        print(f"[{tag}] toy 模式 — 无法观察真实 cache，跳过")
        return
    t0 = time.time()
    resp = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[{"role": "system", "content": LONG_PREFIX},
                  {"role": "user", "content": "给 users 表 email 列加索引，写出 SQL"}],
        max_tokens=100,
    )
    usage = resp.usage
    cached = getattr(usage, "prompt_cache_hit_tokens", "N/A")
    miss = getattr(usage, "prompt_cache_miss_tokens", "N/A")
    print(f"[{tag}] {time.time()-t0:.2f}s | prompt={usage.prompt_tokens} cached={cached} miss={miss}")

timed_call("第 1 次（冷）")
timed_call("第 2 次（应命中）")

[第 1 次（冷）] 1.83s | prompt=671 cached=0 miss=671


[第 2 次（应命中）] 1.57s | prompt=671 cached=640 miss=31


## 工程边界

- 隔离的收益 = sub-agent 中间过程 token × 任务数；代价 = 切片选择错误时 sub-agent 缺上下文
- Caching 不改变质量，只改变价格与延迟 — 但设计消息结构时必须「缓存感知」
- Tool result 裁剪：返回给模型的应是「结论 + 关键行」，原始 10KB 日志留在工具侧供引用

## 练习

1. 修改 `orchestrator`：把切片逻辑换成让 LLM 自己选择需要哪个切片（两轮调用：先选切片再回答），对比 token 总量。
2. 在第 5 个 cell 的 `LONG_PREFIX` 开头加一行 `f"当前时间：{time.time()}"`，重跑两次调用，观察 cache 命中率如何崩塌。
3. 设计一个多 sub-agent 系统：researcher / coder / reviewer 各自需要主 context 的哪部分？画出数据流图。